# CareLoop Clinics — No-Show Guard
## Part 1: Python & Data Foundations (Ingest, Validate, Clean, Enrich)

### Task 1.1 — Read the file the foundations way (`csv` module)

In [ ]:
import csv
import json

def parse_row(row):
    converted = dict(row)
    converted["age"] = int(row["age"])
    converted["fee_inr"] = int(row["fee_inr"])
    converted["prior_no_shows"] = int(row["prior_no_shows"])
    converted["distance_km"] = float(row["distance_km"])
    return converted

accepted = []
rejected = []

with open("data/appointments.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    header = reader.fieldnames
    rows = list(reader)

print("Header:", header)
print("Total rows read:", len(rows))

for row in rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)

with open("rejected_rows.json", "w", encoding="utf-8") as f:
    json.dump(rejected, f, indent=2)

print(f"Accepted: {len(accepted)} | Rejected: {len(rejected)}")

### Task 1.2 — Inspect and clean, pass 1 (`pandas`)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/appointments.csv")
print("Raw shape:", df.shape)
print(df.info())
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("Raw distinct booking_channel:", df["booking_channel"].nunique())

# (a) Drop exact duplicate rows
df = df.drop_duplicates()
print("After dropping duplicates:", len(df))

# (b) Clean booking_channel
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("Cleaned booking_channel values:", sorted(df["booking_channel"].unique()))

# (c) Convert appt_date and booked_on
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("After date conversion:", len(df))

# (d) Drop rows where appt_date or status is missing
df = df.dropna(subset=["appt_date", "status"]).copy()
print("After dropping missing appt_date/status:", len(df))

### Task 1.3 — Clean, pass 2: validity, gaps and outliers

In [ ]:
# (a) Treat age < 1 or age > 100 as invalid
invalid_age_mask = (df["age"] < 1) | (df["age"] > 100)
print("Invalid ages set to missing:", invalid_age_mask.sum())
df.loc[invalid_age_mask, "age"] = np.nan
print("Total missing ages:", df["age"].isnull().sum())

# (b) Fill missing age and distance_km with medians
age_median = df["age"].median()
dist_median = df["distance_km"].median()
print(f"Median age: {age_median} | Median distance_km: {dist_median}")

df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(dist_median)

# (c) IQR outlier capping for distance_km and fee_inr
for col in ["distance_km", "fee_inr"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_capped = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = np.clip(df[col], lower, upper)
    print(f"{col}: Q1={q1}, Q3={q3}, IQR={iqr}, fences=[{lower}, {upper}], capped={n_capped}")

# (d) Round distance_km to 1 decimal and fee_inr to whole rupees
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print("Largest fee_inr after rounding:", df["fee_inr"].max())

### Task 1.4 — Features and a NumPy summary

In [ ]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

print(f"lead_days range: {df['lead_days'].min()} to {df['lead_days'].max()}")
print(f"No-shows: {df['no_show'].sum()} / {len(df)} ({100 * df['no_show'].mean():.1f}%)")

X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# (a) Column means
col_means = np.mean(X, axis=0)
print("Column means [age, distance_km, fee_inr, lead_days]:", np.round(col_means, 2))

# (b) lead_days > 14 using boolean mask
lead_mask = X[:, 3] > 14
print(f"lead_days > 14: {lead_mask.sum()} rows ({100 * lead_mask.mean():.1f}%)")

# (c) z-scores of distance_km and count with |z| > 2
dist_col = X[:, 1]
z_dist = (dist_col - np.mean(dist_col)) / np.std(dist_col)
print("Rows with |z| > 2 for distance_km:", (np.abs(z_dist) > 2).sum())

### Task 1.5 — Enrich the clinics from a live API (Open-Meteo)

In [ ]:
import requests

clinics_df = pd.read_csv("data/clinics.csv")
temps = []

for _, row in clinics_df.iterrows():
    temp_val = None
    reason = None
    for attempt in range(2):
        try:
            resp = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params={
                    "latitude": row["latitude"],
                    "longitude": row["longitude"],
                    "hourly": "temperature_2m",
                    "forecast_days": 1,
                },
                timeout=10,
            )
            if resp.status_code == 200:
                payload = resp.json()
                hourly_temps = payload.get("hourly", {}).get("temperature_2m")
                if hourly_temps and len(hourly_temps) > 0:
                    temp_val = hourly_temps[0]
                    reason = None
                    break
                else:
                    reason = "Missing hourly.temperature_2m in JSON"
            else:
                reason = f"HTTP status {resp.status_code}"
        except requests.exceptions.RequestException as e:
            reason = str(e)
    if temp_val is None:
        print(f"API failed for {row['clinic_name']}: {reason}")
    else:
        print(f"{row['clinic_name']}: {temp_val} C")
    temps.append(temp_val)

clinics_df["temp_c"] = temps
clinics_df.to_csv("clinics_enriched.csv", index=False)
print("Saved clinics_enriched.csv with", len(clinics_df), "rows.")

### Task 1.6 — Export the clean table and write `load_data.sql`

In [ ]:
output_cols = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "age", "gender", "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows", "status",
    "lead_days", "no_show"
]

df = df.sort_values("appointment_id").reset_index(drop=True)
df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")
df = df[output_cols]

print("Unique appointment_id count:", df["appointment_id"].nunique())
print("Total missing values across 16 columns:", df.isnull().sum().sum())

df.to_csv("appointments_clean.csv", index=False)

with open("load_data.sql", "w", encoding="utf-8") as f:
    for _, r in clinics_df.iterrows():
        temp_sql = "NULL" if pd.isna(r["temp_c"]) else f"{float(r['temp_c']):.1f}"
        f.write(
            f"INSERT INTO clinics VALUES ('{r['clinic_id']}','{r['clinic_name']}','{r['city']}',"
            f"'{r['tier']}',{r['latitude']},{r['longitude']},{temp_sql});\n"
        )
    for _, r in df.iterrows():
        f.write(
            f"INSERT INTO appointments VALUES ('{r['appointment_id']}','{r['patient_id']}','{r['clinic_id']}',"
            f"'{r['booked_on']}','{r['appt_date']}',{int(r['age'])},'{r['gender']}',{float(r['distance_km']):.1f},"
            f"{int(r['fee_inr'])},'{r['booking_channel']}','{r['visit_type']}','{r['reminder_sent']}',"
            f"{int(r['prior_no_shows'])},'{r['status']}',{int(r['lead_days'])},{int(r['no_show'])});\n"
        )

print("Saved appointments_clean.csv and load_data.sql.")